In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import transforms, datasets
from torch.utils.data import DataLoader
from sklearn.cluster import KMeans
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt
import numpy as np
import os
from PIL import Image
import seaborn as sns
from tqdm import tqdm

# 设置随机种子确保可复现性
torch.manual_seed(42)
np.random.seed(42)

# 检查CUDA可用性
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"使用设备: {device}")

# 创建结果目录
os.makedirs('unsupervised_results', exist_ok=True)

# 针对1600x350尺寸的预处理
transform = transforms.Compose([
    transforms.Resize((350, 1600)),
    transforms.CenterCrop((320, 1536)),
    transforms.Resize((80, 384)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# 加载数据集
dataset = datasets.ImageFolder('images', transform=transform)
dataloader = DataLoader(dataset, batch_size=32, shuffle=False, num_workers=4)

# 定义卷积自编码器
class ConvAutoencoder(nn.Module):
    def __init__(self):
        super(ConvAutoencoder, self).__init__()
        
        # 编码器
        self.encoder = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=(3, 7), padding=(1, 3)),
            nn.ReLU(),
            nn.MaxPool2d((2, 4)),  # 输出: [32, 40, 96]
            
            nn.Conv2d(32, 64, kernel_size=(3, 7), padding=(1, 3)),
            nn.ReLU(),
            nn.MaxPool2d((2, 4)),  # 输出: [64, 20, 24]
            
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d((2, 2)),  # 输出: [128, 10, 12]
        )
        
        # 解码器
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(128, 64, kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.ReLU(),
            
            nn.ConvTranspose2d(64, 32, kernel_size=(3, 7), stride=(2, 4), padding=(1, 3), output_padding=(1, 3)),
            nn.ReLU(),
            
            nn.ConvTranspose2d(32, 3, kernel_size=(3, 7), stride=(2, 4), padding=(1, 3), output_padding=(1, 3)),
            nn.Sigmoid()  # 输出值在0-1之间
        )
    
    def forward(self, x):
        x = self.encoder(x)
        x = self.decoder(x)
        return x
    
    def encode(self, x):
        with torch.no_grad():
            return self.encoder(x)

# 初始化模型和优化器
model = ConvAutoencoder().to(device)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# 训练自编码器
num_epochs = 50
train_losses = []

print("训练自编码器中...")
for epoch in range(num_epochs):
    running_loss = 0.0
    for images, _ in tqdm(dataloader, desc=f"Epoch {epoch+1}/{num_epochs}"):
        images = images.to(device)
        
        # 前向传播
        outputs = model(images)
        loss = criterion(outputs, images)
        
        # 反向传播和优化
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * images.size(0)
    
    epoch_loss = running_loss / len(dataloader.dataset)
    train_losses.append(epoch_loss)
    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {epoch_loss:.6f}")

# 保存自编码器模型
torch.save(model.state_dict(), 'unsupervised_results/autoencoder.pth')

# 绘制训练损失曲线
plt.figure(figsize=(10, 6))
plt.plot(train_losses)
plt.title('自编码器训练损失')
plt.xlabel('Epoch')
plt.ylabel('MSE Loss')
plt.grid(True)
plt.savefig('unsupervised_results/autoencoder_loss.png', dpi=300)
plt.close()

# 提取所有图像的潜在特征
print("\n提取图像特征...")
features = []
image_paths = [item[0] for item in dataset.imgs]
true_labels = [item[1] for item in dataset.imgs]

model.eval()
with torch.no_grad():
    for images, _ in tqdm(dataloader):
        images = images.to(device)
        batch_features = model.encode(images)
        features.append(batch_features.cpu().numpy())

# 展平特征并转换为numpy数组
features = np.vstack(features)
features = features.reshape(features.shape[0], -1)  # 展平每个样本的特征
print(f"提取的特征形状: {features.shape}")

# 使用K-means聚类
print("\n进行K-means聚类...")
kmeans = KMeans(n_clusters=5, random_state=42, n_init=10)
cluster_labels = kmeans.fit_predict(features)

# 保存聚类结果
np.save('unsupervised_results/cluster_labels.npy', cluster_labels)
np.save('unsupervised_results/true_labels.npy', true_labels)

# 评估聚类结果
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score

ari = adjusted_rand_score(true_labels, cluster_labels)
nmi = normalized_mutual_info_score(true_labels, cluster_labels)
print(f"调整兰德指数 (ARI): {ari:.4f}")
print(f"标准化互信息 (NMI): {nmi:.4f}")

# 使用t-SNE可视化聚类结果
print("\n使用t-SNE降维可视化...")
tsne = TSNE(n_components=2, random_state=42, perplexity=30)
features_2d = tsne.fit_transform(features)

plt.figure(figsize=(12, 10))
sns.scatterplot(x=features_2d[:, 0], y=features_2d[:, 1], 
                hue=cluster_labels, palette='viridis', 
                s=50, alpha=0.8, edgecolor='none')
plt.title('t-SNE可视化聚类结果')
plt.savefig('unsupervised_results/tsne_clusters.png', dpi=300)
plt.close()

# 绘制真实标签与聚类标签的对比
plt.figure(figsize=(14, 6))

plt.subplot(1, 2, 1)
sns.scatterplot(x=features_2d[:, 0], y=features_2d[:, 1], 
                hue=true_labels, palette='viridis', 
                s=50, alpha=0.8, edgecolor='none')
plt.title('真实标签分布')

plt.subplot(1, 2, 2)
sns.scatterplot(x=features_2d[:, 0], y=features_2d[:, 1], 
                hue=cluster_labels, palette='viridis', 
                s=50, alpha=0.8, edgecolor='none')
plt.title('聚类标签分布')

plt.tight_layout()
plt.savefig('unsupervised_results/true_vs_cluster.png', dpi=300)
plt.close()

# 可视化聚类中心图像
print("\n可视化聚类中心...")
cluster_centers = kmeans.cluster_centers_

# 找到每个聚类的代表性图像（最接近中心的图像）
representative_indices = []
for i in range(5):
    # 计算所有样本到聚类中心的距离
    distances = np.linalg.norm(features - cluster_centers[i], axis=1)
    # 找到距离最小的10个样本
    closest_indices = np.argsort(distances)[:10]
    representative_indices.append(closest_indices)

# 可视化聚类结果
plt.figure(figsize=(15, 12))
for cluster_idx in range(5):
    # 显示聚类标签
    plt.subplot(5, 1, cluster_idx+1)
    plt.text(0.5, 0.5, f'Cluster {cluster_idx}', 
             fontsize=16, ha='center', va='center')
    plt.axis('off')
    
    # 显示该聚类的代表性图像
    plt.figure(figsize=(15, 3))
    plt.suptitle(f'Cluster {cluster_idx} - 代表性图像', fontsize=16)
    
    for j, img_idx in enumerate(representative_indices[cluster_idx][:5]):
        img_path = image_paths[img_idx]
        img = Image.open(img_path)
        
        plt.subplot(1, 5, j+1)
        plt.imshow(img)
        plt.title(f'Img {img_idx}')
        plt.axis('off')
    
    plt.tight_layout()
    plt.savefig(f'unsupervised_results/cluster_{cluster_idx}_examples.png', dpi=300)
    plt.close()

# 生成聚类报告
from sklearn.metrics import confusion_matrix, classification_report

# 创建聚类到真实标签的映射（多数投票）
cluster_to_label = {}
for cluster in range(5):
    # 获取该聚类中的所有真实标签
    true_in_cluster = [true_labels[i] for i, c in enumerate(cluster_labels) if c == cluster]
    # 找到最常见的真实标签
    most_common = max(set(true_in_cluster), key=true_in_cluster.count)
    cluster_to_label[cluster] = most_common

print("\n聚类到真实标签的映射:", cluster_to_label)

# 将聚类标签映射到真实标签
predicted_labels = [cluster_to_label[c] for c in cluster_labels]

# 计算混淆矩阵
cm = confusion_matrix(true_labels, predicted_labels)

# 绘制混淆矩阵
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=dataset.classes, 
            yticklabels=dataset.classes)
plt.title('聚类结果混淆矩阵')
plt.xlabel('预测标签')
plt.ylabel('真实标签')
plt.savefig('unsupervised_results/confusion_matrix.png', dpi=300)
plt.close()

# 打印分类报告
print("\n聚类分类报告:")
print(classification_report(true_labels, predicted_labels, target_names=dataset.classes))

# 可视化重建效果
print("\n可视化自编码器重建效果...")
test_images = next(iter(dataloader))[0][:5]  # 取前5张图像
test_images = test_images.to(device)

model.eval()
with torch.no_grad():
    reconstructed = model(test_images)

# 反标准化函数
def denormalize(tensor):
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1).to(device)
    std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1).to(device)
    return tensor * std + mean

# 可视化原始和重建图像
plt.figure(figsize=(15, 6))
for i in range(5):
    # 原始图像
    plt.subplot(2, 5, i+1)
    img = denormalize(test_images[i]).cpu().permute(1, 2, 0).numpy()
    plt.imshow(np.clip(img, 0, 1))
    plt.title(f'原始 {i}')
    plt.axis('off')
    
    # 重建图像
    plt.subplot(2, 5, i+6)
    rec_img = denormalize(reconstructed[i]).cpu().permute(1, 2, 0).numpy()
    plt.imshow(np.clip(rec_img, 0, 1))
    plt.title(f'重建 {i}')
    plt.axis('off')

plt.tight_layout()
plt.savefig('unsupervised_results/reconstruction_examples.png', dpi=300)
plt.close()

print("\n非监督学习完成! 所有结果保存在 'unsupervised_results' 目录")